# Análisis Exploratorio de Datos (EDA) - Consumo de Energía en Latinoamérica

Este notebook documenta la exploración inicial de los datos de *Our World in Data (World Energy Consumption)*, filtrando por los países de América Latina desde el año 2000 en adelante. El objetivo es justificar la selección de variables y la narrativa del dashboard.

In [ ]:
import pandas as pd
import numpy as np

# Cargar el dataset desde la nueva ubicación del proyecto
file_path = "../data/owid-energy-data (1).csv"
df = pd.read_csv(file_path)
print(f"Filas totales en el dataset original: {df.shape[0]}")
print(f"Columnas totales: {df.shape[1]}")

## 1. Filtrado para América Latina (LAC) desde el Año 2000

In [ ]:
lac_countries = [
    'Argentina', 'Bolivia', 'Brazil', 'Chile', 'Colombia', 'Costa Rica',
    'Cuba', 'Dominican Republic', 'Ecuador', 'El Salvador', 'Guatemala',
    'Haiti', 'Honduras', 'Mexico', 'Nicaragua', 'Panama', 'Paraguay',
    'Peru', 'Uruguay', 'Venezuela'
]

# Filtrar dataset
df_lac = df[(df['country'].isin(lac_countries)) & (df['year'] >= 2000)].copy()
print(f"Registros de América Latina desde el 2000: {df_lac.shape[0]}")
df_lac['country'].value_counts()

## 2. Análisis de Cobertura y Valores Nulos de las Variables Recomendadas

Evaluaremos la disponibilidad de datos para las variables clave sugeridas en la guía de práctica.

In [ ]:
cols_key = [
    'country', 'year', 'gdp', 'energy_per_capita', 
    'renewables_share_energy', 'renewables_share_elec', 
    'carbon_intensity_elec', 'greenhouse_gas_emissions'
]

# Ver porcentaje de datos no nulos por columna clave
missing_summary = df_lac[cols_key].isnull().mean() * 100
coverage_summary = 100 - missing_summary

df_coverage = pd.DataFrame({
    'Datos Existentes (%)': coverage_summary,
    'Datos Faltantes (%)': missing_summary
})
df_coverage

### Hallazgo Importante:
*   `renewables_share_energy` (participación de renovables en la energía primaria total) solo tiene un **39.0%** de cobertura en América Latina.
*   `renewables_share_elec` (participación de renovables en la generación eléctrica) tiene un **100%** de cobertura.
*   **Decisión**: Utilizaremos `renewables_share_elec` como métrica principal para la transición verde, ya que permite comparar a todos los países sin sesgos por datos perdidos.

## 3. Análisis de Desacoplamiento (Renovables vs. Intensidad de Carbono)

Exploramos si existe una correlación inversa entre la adopción de energías renovables y la intensidad de CO2 emitida en la generación eléctrica.

In [ ]:
# Correlación lineal de Pearson en el subconjunto de LAC
corr_matrix = df_lac[['renewables_share_elec', 'carbon_intensity_elec', 'gdp']].corr()
corr_matrix

Vemos una **fuerte correlación negativa** entre `renewables_share_elec` y `carbon_intensity_elec`. A mayor porcentaje de electricidad renovable, menor es la intensidad de carbono (gCO2 por kWh). Esto justifica la estructura de nuestro Gráfico 1 (subplot temporal).

## 4. Clasificación de Líderes vs. Rezagados al 2022

In [ ]:
# Filtrar para el año más reciente con datos robustos (2022)
df_2022 = df_lac[df_lac['year'] == 2022].sort_values(by='renewables_share_elec', ascending=False)

# Clasificar países
def clasificar(val):
    if val > 70: return 'Líder Verde (>70%)'
    elif val >= 35: return 'Transición (35-70%)'
    else: return 'Rezagado Fósil (<35%)'

df_2022['categoria'] = df_2022['renewables_share_elec'].apply(clasificar)
df_2022[['country', 'renewables_share_elec', 'carbon_intensity_elec', 'categoria']]